In [1]:
# @title Package
from natsort import natsorted
import numpy as np
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchaudio
import math
from sklearn import svm

import torchvision
import torchvision.transforms as transforms
import torchaudio.models as audio_models

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

import time

lib_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44/Code'
os.chdir(lib_dir)
print('library directory: ' + lib_dir)
from modules.networks_clf import *
from modules.signal import spectro_norm, lfp_spectro
from modules.data import *
from modules.metrics import accu_fun

library directory: /content/drive/MyDrive/Project/BrainRegionId/Project44/Code


In [2]:
# @title Load device
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

!nvidia-smi -L


/bin/bash: line 1: nvidia-smi: command not found


In [3]:
!pip install networkx
import networkx as nx

In [4]:
# Set the signal parameters
spectro_args = {
    'nfft':800,
    'power':1,
    'LFP_bound':[0, 500],
    'MUA_bound':[500, 2000],
    'spectro_img':[224, 28],
    'LFP_img':[56 * 4, 28],
    'MUA_img':[0, 28],
    'sampling_lfp':2500,
    'sampling_mua':5000,
    'Log':False,
}

dict_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project37/Data/dat'
acronym_list = acronym_list_gen(dict_dir)

In [5]:
Mat_dict_stim = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/Confusion_mat_stimOn_times.pt', weights_only=False)
Mat_dict_rest = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Project44/Result/Confusion_mat.pt', weights_only=False)

In [6]:
community_dict_rest = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/community_dict.pt', weights_only=False)

In [7]:
community_dict_stim = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/StimOn/community_stimOn_dict.pt', weights_only=False)

In [8]:
################################################################################
import subprocess
import sys
required = {'ONE-api', 'brain', 'ibllib'}
subprocess.check_call([sys.executable, '-m', 'pip', 'install', *required])

from one.api import ONE
from brainbox.io.one import SessionLoader, SpikeSortingLoader

from iblatlas.atlas import AllenAtlas

ba = AllenAtlas()
br = ba.regions
ba.compute_regions_volume()

Downloading: /root/Downloads/ONE/openalyx.internationalbrainlab.org/histology/ATLAS/Needles/Allen/average_template_25.nrrd Bytes: 32998960


100%|██████████| 31.470260620117188/31.470260620117188 [00:00<00:00, 297.96it/s]


Downloading: /root/Downloads/ONE/openalyx.internationalbrainlab.org/histology/ATLAS/Needles/Allen/annotation_25.nrrd Bytes: 4035363


100%|██████████| 3.848422050476074/3.848422050476074 [00:00<00:00, 250.35it/s]


In [9]:
len(br.acronym)

2655

In [10]:
len(br.name)

2655

In [11]:
import copy
resolution = 1.0

In [12]:
degree_dat = {}
for name in ['AnyNet', 'ViT', 'RNN']:
    G = nx.from_numpy_array(copy.deepcopy(Mat_dict_rest[name]))
    degree_dat[name] = G.degree()

layer_counts = {}
hub_acronyms = []
for comm in np.unique(community_dict_rest[resolution]['communities_label']):
    comm_acronyms = community_dict_rest[resolution]['communities_acronym'][np.argwhere(community_dict_rest[resolution]['communities_label'] == comm).flatten()]
    comm_ids = community_dict_rest[resolution]['communities'][np.argwhere(community_dict_rest[resolution]['communities_label'] == comm).flatten()]
    comm_degrees = []
    for id in comm_ids:
        comm_degrees.append(max(degree_dat['AnyNet'][id], degree_dat['ViT'][id], degree_dat['RNN'][id]))
    hub_acronyms.append(comm_acronyms[np.argmax(np.array(comm_degrees)).flatten()])
    print(f'community label: {comm}')
    print(f'community hub: {comm_acronyms[np.argmax(np.array(comm_degrees)).flatten()]}')
    print(f'community member: {comm_acronyms}')

hub_acronyms_rest = np.array(hub_acronyms)

community label: 0
community hub: ['CUL4 5']
community member: ['KF' 'I5' 'DCO' 'VCO' 'CU' 'ECU' 'SPVC' 'SPVI' 'Pa5' 'AMBv' 'GRN' 'ICB'
 'IO' 'IRN' 'LIN' 'LRNm' 'MARN' 'MDRNd' 'MDRNv' 'PARN' 'PAS' 'PGRNd'
 'PGRNl' 'PRP' 'LAV' 'MV' 'SPIV' 'x' 'y' 'FN' 'IP' 'DN' 'VeCB' 'NTS'
 'SPVO' 'LING' 'CENT2' 'CENT3' 'CUL4 5' 'DEC' 'FOTU' 'PYR' 'UVU' 'NOD'
 'SIM' 'ANcr1' 'ANcr2' 'PRM' 'COPY' 'PFL' 'FL']
community label: 1
community hub: ['CUN']
community member: ['ICc' 'CUN' 'SOCl' 'PRNc' 'SUT' 'PC5']
community label: 2
community hub: ['CP']
community member: ['FRP6a' 'MOp5' 'MOp6a' 'MOp6b' 'MOs6a' 'MOs6b' 'SSp-m2/3' 'SSp-m4'
 'SSp-m5' 'SSp-m6a' 'SSp-m6b' 'GU5' 'GU6a' 'ACAd5' 'ACAd6a' 'ACAd6b'
 'ACAv2/3' 'ACAv5' 'ACAv6a' 'ACAv6b' 'PL5' 'PL6a' 'PL6b' 'ILA1' 'ILA2/3'
 'ILA5' 'ILA6a' 'ORBl6a' 'ORBm5' 'ORBm6a' 'ORBvl6a' 'AId5' 'AId6a' 'AId6b'
 'AIp6b' 'AIv2/3' 'AIv5' 'AIv6a' 'AIv6b' 'CLA' 'EPd' 'CP' 'ACB']
community label: 3
community hub: ['VISp5']
community member: ['VISp1' 'VISp2/3' 'VISp4' 'VISp5' '

In [13]:
degree_dat = {}
for name in ['AnyNet', 'ViT', 'RNN']:
    G = nx.from_numpy_array(copy.deepcopy(Mat_dict_stim[name]))
    degree_dat[name] = G.degree()

layer_counts = {}
hub_acronyms = []
for comm in np.unique(community_dict_stim[resolution]['communities_label']):
    comm_acronyms = community_dict_stim[resolution]['communities_acronym'][np.argwhere(community_dict_stim[resolution]['communities_label'] == comm).flatten()]
    comm_ids = community_dict_stim[resolution]['communities'][np.argwhere(community_dict_stim[resolution]['communities_label'] == comm).flatten()]
    comm_degrees = []
    for id in comm_ids:
        comm_degrees.append(max(degree_dat['AnyNet'][id], degree_dat['ViT'][id], degree_dat['RNN'][id]))
    hub_acronyms.append(comm_acronyms[np.argmax(np.array(comm_degrees)).flatten()])
    print(f'community label: {comm}')
    print(f'community hub: {comm_acronyms[np.argmax(np.array(comm_degrees)).flatten()]}')
    print(f'community member: {comm_acronyms}')

hub_acronyms_stim = np.array(hub_acronyms)

community label: 0
community hub: ['SSp-bfd2/3']
community member: ['SSp-bfd2/3' 'TEa2/3' 'TEa4']
community label: 1
community hub: ['CP']
community member: ['FRP5' 'FRP6a' 'MOs5' 'MOs6a' 'MOs6b' 'SSp-n2/3' 'VISC6b' 'ACAd5'
 'ACAd6a' 'ACAd6b' 'ACAv1' 'ACAv2/3' 'ACAv5' 'ACAv6a' 'ACAv6b' 'PL6a'
 'PL6b' 'ILA6a' 'AIp6b' 'IG' 'CP']
community label: 2
community hub: ['MOp6a']
community member: ['MOp5' 'MOp6a' 'MOp6b' 'SSp-n4' 'SSp-n6a' 'SSp-n6b' 'SSp-m6a' 'SSp-m6b'
 'ORBl5' 'ORBl6a' 'AId6a' 'AId6b' 'AIv5' 'AIv6a' 'AIv6b' 'CLA']
community label: 3
community hub: ['LSr']
community member: ['LSc' 'LSr' 'LSv' 'SF' 'SH' 'TRS' 'SFO' 'BST']
community label: 4
community hub: ['ORBvl6a']
community member: ['PL5' 'ILA1' 'ILA2/3' 'ILA5' 'ORBm5' 'ORBm6a' 'ORBvl6a']
community label: 5
community hub: ['VISpm5']
community member: ['VISpm5' 'VISpm6a' 'VISpm6b' 'RSPd6b' 'RSPv6b']
community label: 6
community hub: ['VISp2/3']
community member: ['SSs2/3' 'VISp2/3' 'VISpl6a' 'RSPv6a']
community label: 7
communi

In [19]:
resolution = 1.0
br_dict = {}
br_list = []
br_name = []
Cosmos_name = []
community_stimOff_list = []
community_stimOn_list = []
Cosmos_list = []
hub_list_rest = []
hub_list_stim = []
for acronym in acronym_list:
    br_list.append(acronym)
    Cosmos_acronym = br.acronym2acronym(acronym, mapping='Cosmos')[0]
    Cosmos_list.append(Cosmos_acronym)
    br_name.append(br.name[np.argwhere(br.acronym == acronym)[0]].item())
    Cosmos_name.append(br.name[np.argwhere(br.acronym == Cosmos_acronym)[0]].item())
    if acronym in community_dict_rest[resolution]['communities_acronym']:
        community_stimOff_list.append(community_dict_rest[resolution]['communities_label'][np.argwhere(community_dict_rest[resolution]['communities_acronym'] == acronym).flatten()].item())
        hub_list_rest.append(hub_acronyms_rest[community_dict_rest[resolution]['communities_label'][np.argwhere(community_dict_rest[resolution]['communities_acronym'] == acronym).flatten()].item()].item())
    else:
        community_stimOff_list.append(-1)
        hub_list_rest.append('N/A')
    if acronym in community_dict_stim[resolution]['communities_acronym']:
        community_stimOn_list.append(community_dict_stim[resolution]['communities_label'][np.argwhere(community_dict_stim[resolution]['communities_acronym'] == acronym).flatten()].item())
        hub_list_stim.append(hub_acronyms_stim[community_dict_stim[resolution]['communities_label'][np.argwhere(community_dict_stim[resolution]['communities_acronym'] == acronym).flatten()].item()].item())
    else:
        community_stimOn_list.append(-1)
        hub_list_stim.append('N/A')


br_dict = pd.DataFrame(
    {
        'Allen acronym':br_list,
        'Allen name':br_name,
        'Inter-trial state':community_stimOff_list,
        'Inter-trial state hub':hub_list_rest,
        'Audio-visual cue':community_stimOn_list,
        'Audio-visual cue hub':hub_list_stim,

    }
)

In [20]:
br_dict

,Allen acronym,Allen name,Inter-trial state,Inter-trial state hub,Audio-visual cue,Audio-visual cue hub
0,FRP1,Frontal pole layer 1,10,AON,26,ProS
1,FRP2/3,Frontal pole layer 2/3,10,AON,26,ProS
2,FRP5,Frontal pole layer 5,-1,N/A,1,CP
3,FRP6a,Frontal pole layer 6a,2,CP,1,CP
4,MOp1,Primary motor area Layer 1,-1,N/A,-1,N/A
...,...,...,...,...,...,...
467,ANcr2,Crus 2,0,CUL4 5,9,IRN
468,PRM,Paramedian lobule,0,CUL4 5,9,IRN
469,COPY,Copula pyramidis,0,CUL4 5,9,IRN
470,PFL,Paraflocculus,0,CUL4 5,9,IRN


In [23]:
br_dict.to_csv('/content/drive/MyDrive/Project/BrainRegionId/Nature/results/communities/br_dict2.csv', index=False)

In [22]:
!dir

br_dict2.csv
br_dict.csv
Coordinates_test.ipynb
Copy\ of\ brain_region_id_to_community_mapping.ipynb
Copy\ of\ Deeplift_stat.ipynb
Copy\ of\ LFP_net_training_Allen_rest_acu_analysis.ipynb
Copy\ of\ LFP_net_training_Allen_rest_Captum_DeepLift.ipynb
Copy\ of\ LFP_net_training_Allen_rest_Captum_DeepLift_stat.ipynb
Copy\ of\ LFP_net_training_Allen_rest_Captum.ipynb
Copy\ of\ LFP_net_training_Allen_rest_CommunityMapping.ipynb
Copy\ of\ LFP_net_training_Allen_rest_Confusion_mat_analysis.ipynb
Copy\ of\ LFP_net_training_Allen_rest_Confusion_mat_gen.ipynb
Copy\ of\ LFP_net_training_Allen_rest.ipynb
Copy\ of\ LFP_net_training_Allen_rest_Representation_mat_gen.ipynb
Copy\ of\ LFP_net_training_Allen_rest_VISp.ipynb
Copy\ of\ LFP_net_training_CommunityMapping_rest_acu_analysis.ipynb
Copy\ of\ LFP_net_training_Cosmos_rest_acu_analysis.ipynb
Copy\ of\ Project44_console.ipynb
Copy\ of\ Representation_Analysis.ipynb
Copy\ of\ SupplementaryFig.ipynb
LFP_net_training_Allen_rest_acu_analysis_20250306.ipy

In [ ]:
br.name[np.argwhere(br.acronym == 'CA1')[0]].item()

'Field CA1'

In [ ]:
Cosmos_list

['Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isoc